# Computing bijective correspondences by minimizing the Ginzburg-Landau energy

This notebook showcases the basic usage of the method: given two genus-zero surfaces
$A$ and $B$ and a (possibly low-quality) input map between them, we compute a
continuous, bijective, orientation-preserving correspondence.

The key idea is that a bijection $\varphi : A \to B$ can be encoded *implicitly* as the
zero set of a **complex section** $z$ on the product space $A \times B$. Minimizing the
**Ginzburg-Landau (GL) energy** of $z$ shrinks the area of that zero set.

The algorithm has four steps, mirrored below:

1. **Surface connections** — build a discrete connection on each surface with total
   curvature $2\pi$, so that each slice of the product space carries a single zero.
2. **Initialization** — turn the input map into an initial section $z$ on $A \times B$.
3. **GL minimization** — minimize the discrete Ginzburg-Landau energy of $z$.
4. **Evaluation** — read off the correspondence from the zero set of the optimized $z$.

Each mechanical step is packaged as a helper in `diffops.pipeline`; all plotting is
routed through `plot_utils` (an external, user-owned module).

## Setup

In [ ]:
import os
import sys

# ScalableDenseMaps ships with this repo and provides the precise-map data structure
# used to represent and apply correspondences.
sys.path.append(os.path.abspath("./ScalableDenseMaps/"))

import numpy as np
import potpourri3d as pp3d

import diffops
import diffops.pipeline as pipeline
import densemaps.numpy.maps as maps_np

# All visualization goes through this thin, user-owned wrapper (see plot_utils.py),
# which keeps the external viztools/pyFM dependencies out of the notebook body.
import plot_utils as plu

## Load the input meshes

We read the two surfaces $A$ and $B$, then convert them to `SurfaceMesh` objects and
compute an **intrinsic Delaunay** triangulation. The intrinsic retriangulation improves
the conditioning of the connection Laplacian without moving any vertex, so it does not
alter the geometry of the input.

In [ ]:
# NOTE: these are machine-specific paths to a sample mesh pair. Point them at your
# own data (the repository does not ship a `data/` folder).
# mesh_path_A = "/Users/robinmagnet/Desktop/Postdoc/Projets/minimal_surf_higher/data/cow_horse_5/A.obj"
# mesh_path_B = "/Users/robinmagnet/Desktop/Postdoc/Projets/minimal_surf_higher/data/cow_horse_5/B.obj"

mesh_path_A = "./data/sitting_armadillo/A_1.obj"
mesh_path_B = "./data/sitting_armadillo/B_1.obj"


v1, f1 = pp3d.read_mesh(mesh_path_A)
v2, f2 = pp3d.read_mesh(mesh_path_B)

In [ ]:
mesh1 = diffops.SurfaceMesh(v1.copy(), f1.copy())
mesh1.make_delaunay(verbose=True)

mesh2 = diffops.SurfaceMesh(v2.copy(), f2.copy())
mesh2.make_delaunay(verbose=True)

In [ ]:
mesh1.n_vertices, mesh2.n_vertices

In [ ]:
plu.plot(mesh1)

## Input correspondence

The method starts from an input map given as **vertex-to-face maps** in both directions.
Here we use a trivial nearest-neighbor (in ambient space) initialization on the original
triangulations.

In [ ]:
# Vertex-to-face precise maps on the ORIGINAL triangulations.
P21_nn = maps_np.EmbPreciseMap(mesh1.vertices, mesh2.vertices, f1)
P12_nn = maps_np.EmbPreciseMap(mesh2.vertices, mesh1.vertices, f2)

In [ ]:
# Visualize the raw input map (mesh2 pushed onto mesh1's connectivity).
# plu.plot(plu.ToyMesh(P12_nn @ mesh2.vertices, f1))
plu.plot_p2p(mesh1, mesh2, P21_nn)

In [ ]:
# Convert each precise map into a per-face intrinsic vector field, the representation
# expected when building the initial section.
v2f_21 = pipeline.precise_map_to_vfield(mesh1, P21_nn.v2f_21, P21_nn.bary_coords)
v2f_12 = pipeline.precise_map_to_vfield(mesh2, P12_nn.v2f_21, P12_nn.bary_coords)

## Step (1): Construct surface connections

On each surface we build a discrete connection whose curvature is **half the Gaussian
curvature** (which integrates to $2\pi$ by Gauss-Bonnet). Fixing this topological class
is what forces exactly one zero per slice of $A \times B$, i.e. a bijection.

In [ ]:
curvature1, connection1 = pipeline.compute_surface_connection(mesh1)
curvature2, connection2 = pipeline.compute_surface_connection(mesh2)

# Convenient bundles to pass around the per-surface data.
connections = (connection1, connection2)
curvatures = (curvature1, curvature2)

## Step (2): Initialize the section

We assemble the product mesh $A \times B$ and compute an initial complex section whose
zero set approximates the graph of the input map. To do so, we build a connection that
concentrates its curvature along the input correspondence and take the smallest
eigenvector of the associated Laplacian.

In [ ]:
pmesh = diffops.ProductMesh(mesh1, mesh2)

min_section, connection = pmesh.compute_initial_section(
    v2f_21=v2f_21,
    v2f_12=v2f_12,
    con_angle_init_1=connection1,
    con_angle_init_2=connection2,
    curv_init_1=curvature1,
    curv_init_2=curvature2,
    use_preconditioner=True,
    verbose=True,
)

In [ ]:
# Extract and visualize the correspondence encoded by the initial section, to see the
# starting point of the optimization.
image_pts_21_init, _, _ = pipeline.section_to_map(
    pmesh, min_section, connections, curvatures, direction="21"
)
P21_init = maps_np.EmbPreciseMap(mesh1.vertices, image_pts_21_init, f1)

# plu.plot(plu.ToyMesh(P21_init @ mesh1.vertices, f2))
plu.plot_p2p(mesh1, mesh2, P21_init)

## Step (3): Ginzburg-Landau minimization

We now minimize the discrete Ginzburg-Landau energy of the section.

The weight `lam` sets the strength of the circular-well potential (larger =
sharper zeros).

In [ ]:
# Assemble the product-space connection Laplacian and scalar mass operators.
Lap_fem, M_scal, (L1, M1, L2, M2) = pipeline.build_bundle_operators(
    mesh1, mesh2, connection1, connection2, curvature1, curvature2
)

In [ ]:
# Critical Ginzburg-Landau parameter from the smallest eigenvalues of L1, L2.
eps_crit = pipeline.critical_epsilon(L1, M1, L2, M2)

In [ ]:
u_optimized, logger = pipeline.run_gl_optimization(
    min_section,
    Lap_fem,
    M_scal,
    eps_crit,
    lam=1e2,  # weight of the circular-well potential
    alpha=1.0,  # weight of the Dirichlet term
    pin_matrix=None,  # no landmarks in this basic example
    n_iter=1000,
    log_interval=50,
    gtol=1e-8,
)

## Step (4): Evaluate the correspondence

Finally, we read the correspondence off the optimized section by locating, for each
vertex, the single zero of $z$ along its slice, and converting that intrinsic location
back to the original triangulation. We extract both directions and visualize the result.

In [ ]:
# Map from A to B and from B to A.
image_pts_21, _, _ = pipeline.section_to_map(
    pmesh, u_optimized, connections, curvatures, direction="21", verbose=True
)
image_pts_12, _, _ = pipeline.section_to_map(
    pmesh, u_optimized, connections, curvatures, direction="12", verbose=True
)

# Wrap the image points in a precise map on the ORIGINAL triangulation.
P21_final = maps_np.EmbPreciseMap(mesh1.vertices, image_pts_21, f1)

In [ ]:
# Visualize the final correspondence.
# plu.plot(plu.ToyMesh(P21_final @ mesh1.vertices, f2))
plu.plot_p2p(
    mesh1, mesh2, P21_final, uv1=mesh1.vertices[:, :2], texture="texture_1.png"
)